# Monocular Depth Estimation — MiDaS
Estimate depth from a single image or video. Results are saved to **Google Drive**.

**Setup:** `Runtime → Change runtime type → GPU (T4)`

In [ ]:
# Mount Google Drive & create output folder
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/depth_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Results will be saved to: {OUTPUT_DIR}')

In [ ]:
# Load model
import torch, cv2, numpy as np, matplotlib.pyplot as plt, time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

MODEL_TYPE = 'DPT_Large'  # Options: DPT_Large, DPT_Hybrid, MiDaS_small

midas = torch.hub.load('intel-isl/MiDaS', MODEL_TYPE)
midas.to(device).eval()

transforms = torch.hub.load('intel-isl/MiDaS', 'transforms')
transform = transforms.dpt_transform if MODEL_TYPE != 'MiDaS_small' else transforms.small_transform
print(f'Model {MODEL_TYPE} loaded.')

In [ ]:
# Depth estimation function
def get_depth(img_bgr):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    input_batch = transform(img_rgb).to(device)
    with torch.no_grad():
        pred = midas(input_batch)
        pred = torch.nn.functional.interpolate(
            pred.unsqueeze(1), size=img_rgb.shape[:2],
            mode='bicubic', align_corners=False
        ).squeeze()
    d = pred.cpu().numpy()
    d = (d - d.min()) / (d.max() - d.min() + 1e-8) * 255
    return d.astype(np.uint8)

---
## Image Depth Estimation
Upload images to Colab using the **file browser** (📁 icon on the left sidebar), then set the path below.

In [ ]:
# Set your image path here
IMAGE_PATH = '/content/your_image.jpg'  # <-- change this

img = cv2.imread(IMAGE_PATH)
assert img is not None, f'Could not read: {IMAGE_PATH}'

start = time.time()
depth = get_depth(img)
print(f'Depth estimated in {time.time()-start:.2f}s')

# Display
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
ax1.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
ax1.set_title('Original'); ax1.axis('off')
im = ax2.imshow(depth, cmap='inferno')
ax2.set_title('Depth Map'); ax2.axis('off')
plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)
plt.tight_layout()

# Save to Drive
name = os.path.splitext(os.path.basename(IMAGE_PATH))[0]
fig.savefig(f'{OUTPUT_DIR}/{name}_result.png', dpi=150, bbox_inches='tight')

sbs = np.hstack([img, cv2.applyColorMap(depth, cv2.COLORMAP_INFERNO)])
cv2.imwrite(f'{OUTPUT_DIR}/{name}_side_by_side.png', sbs)

plt.show()
print(f'Saved to: {OUTPUT_DIR}/{name}_result.png')
print(f'Saved to: {OUTPUT_DIR}/{name}_side_by_side.png')

### Batch: Process All Images in a Folder

In [ ]:
# Process all images in a folder
IMAGE_FOLDER = '/content/images'  # <-- change this (upload a folder or multiple files)

exts = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
if os.path.isdir(IMAGE_FOLDER):
    image_files = sorted([f for f in os.listdir(IMAGE_FOLDER) if f.lower().endswith(exts)])
    print(f'Found {len(image_files)} images')

    for fname in image_files:
        path = os.path.join(IMAGE_FOLDER, fname)
        img = cv2.imread(path)
        if img is None:
            continue

        depth = get_depth(img)
        name = os.path.splitext(fname)[0]

        # Save side-by-side
        sbs = np.hstack([img, cv2.applyColorMap(depth, cv2.COLORMAP_INFERNO)])
        cv2.imwrite(f'{OUTPUT_DIR}/{name}_side_by_side.png', sbs)

        # Save figure
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        ax1.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax1.set_title('Original'); ax1.axis('off')
        ax2.imshow(depth, cmap='inferno'); ax2.set_title('Depth'); ax2.axis('off')
        plt.tight_layout()
        fig.savefig(f'{OUTPUT_DIR}/{name}_result.png', dpi=150, bbox_inches='tight')
        plt.show()
        print(f'✓ {fname}')

    print(f'\nAll saved to: {OUTPUT_DIR}')
else:
    print(f'Folder not found: {IMAGE_FOLDER}')
    print('Upload files using the 📁 sidebar, then update the path.')

---
## Video Depth Estimation
Upload a video to Colab using the **file browser** (📁 sidebar), then set the path below.

In [ ]:
# Set your video path here
VIDEO_PATH = '/content/your_video.mp4'  # <-- change this
SKIP = 1      # 0 = every frame, 1 = every 2nd, 2 = every 3rd
MAX_FRAMES = 200  # set to None for all frames

cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), f'Could not open: {VIDEO_PATH}'

fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
effective_fps = fps / (SKIP + 1)

name = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
out_path = f'{OUTPUT_DIR}/{name}_depth.mp4'

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(out_path, fourcc, effective_fps, (w * 2, h))

print(f'Video: {w}x{h} @ {fps:.0f}fps, {total} frames')
print(f'Processing every {SKIP+1} frame(s), max {MAX_FRAMES or "all"} frames...\n')

idx = 0
done = 0
t0 = time.time()
samples = []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    if SKIP > 0 and idx % (SKIP + 1) != 0:
        idx += 1
        continue
    if MAX_FRAMES and done >= MAX_FRAMES:
        break

    depth = get_depth(frame)
    depth_c = cv2.applyColorMap(depth, cv2.COLORMAP_INFERNO)
    combined = np.hstack([frame, cv2.resize(depth_c, (w, h))])
    out.write(combined)

    if done in [0, total//(2*(SKIP+1)), total//(SKIP+1)-1]:
        samples.append((frame.copy(), depth.copy()))

    done += 1
    idx += 1
    if done % 25 == 0:
        elapsed = time.time() - t0
        print(f'  Frame {done} | {done/elapsed:.1f} FPS')

cap.release()
out.release()

elapsed = time.time() - t0
print(f'\nDone: {done} frames in {elapsed:.1f}s ({done/elapsed:.1f} FPS)')
print(f'Saved to: {out_path}')

# Show sample frames
for i, (f, d) in enumerate(samples):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.imshow(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)); ax1.set_title('Original'); ax1.axis('off')
    ax2.imshow(d, cmap='inferno'); ax2.set_title('Depth'); ax2.axis('off')
    plt.tight_layout()
    fig.savefig(f'{OUTPUT_DIR}/{name}_frame_{i+1}.png', dpi=150, bbox_inches='tight')
    plt.show()

print(f'\nAll results in Google Drive → depth_results/')

---
## List Saved Results

In [ ]:
print(f'Results in: {OUTPUT_DIR}\n')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(f'{OUTPUT_DIR}/{f}') / (1024*1024)
    print(f'  {f:45s} {size:.2f} MB')